In [1]:
import os
from getpass import getpass

os.environ["LLM_PROVIDER"] = "openai"
os.environ["LLM_MODEL"] = "gpt-4.1"
os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key (input is hidden): ")
os.environ["ALLOW_PAID_API"] = "true"
os.environ["EMBEDDING_MODEL"] = ""

if not os.environ["OPENAI_API_KEY"].strip():
    raise ValueError("API key cannot be empty.")

print("API configuration is complete. Please rerun the website launch cell.")

请输入 OpenAI API key（输入会隐藏）： ········


API 配置完成。请重新运行网站启动 cell。


In [5]:
from pathlib import Path
import os
import socket
import subprocess
import sys
import time
import urllib.request
import webbrowser
from IPython.display import display, HTML

# Run this notebook from the project folder containing app.py.
project = Path.cwd()
app_file = project / "app.py"

if not app_file.is_file():
    raise FileNotFoundError(
        f"Cannot find app.py in the current directory: {project}\n"
        "Please switch to the project folder containing app.py."
    )

# Load .env if available, preserving configuration already set in the notebook.
try:
    from dotenv import load_dotenv
    load_dotenv(project / ".env", override=False)
except ImportError:
    pass

# Stop only the previous website process saved by this notebook.
previous = globals().get("website")
if isinstance(previous, subprocess.Popen) and previous.poll() is None:
    previous.terminate()
    try:
        previous.wait(timeout=10)
    except subprocess.TimeoutExpired:
        raise RuntimeError(
            "The previous website process has not stopped. "
            "Please stop it before running this cell again."
        )

# Automatically select an available port.
with socket.socket() as sock:
    sock.bind(("127.0.0.1", 0))
    port = sock.getsockname()[1]

url = f"http://127.0.0.1:{port}"
log_path = project / f"website-{port}.log"

# Start a fresh Python process to avoid reusing previously loaded application code.
with log_path.open("w", encoding="utf-8") as log:
    website = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", str(app_file),
            f"--server.port={port}",
            "--server.address=127.0.0.1",
            "--server.headless=true",
            "--browser.gatherUsageStats=false",
        ],
        cwd=str(project),
        env=os.environ.copy(),
        stdout=log,
        stderr=subprocess.STDOUT,
    )

# Wait for the website's health check before opening the browser.
for _ in range(60):
    if website.poll() is not None:
        raise RuntimeError(
            "Website startup failed:\n"
            + log_path.read_text(encoding="utf-8")
        )

    try:
        with urllib.request.urlopen(
            url + "/_stcore/health", timeout=1
        ) as response:
            ready = response.status == 200
    except OSError:
        ready = False

    if ready:
        print("Project directory:", project)
        print("Website address:", url)
        display(HTML(
            f'<a href="{url}" target="_blank">Open Travel Agent</a>'
        ))
        webbrowser.open_new_tab(url)
        break

    time.sleep(1)
else:
    website.terminate()
    raise RuntimeError(
        "Website startup timed out:\n"
        + log_path.read_text(encoding="utf-8")
    )

运行目录： /Users/chenwang/Desktop/Travel-agent 2
网站地址： http://127.0.0.1:54381
